<a href="https://colab.research.google.com/github/GauriNair05/CNN/blob/main/Image_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
ashfakyeafi_cat_dog_images_for_classification_path = kagglehub.dataset_download('ashfakyeafi/cat-dog-images-for-classification')
mohamedhassanali_apple_vs_tomatoes_binaryclassify_pytorch_path = kagglehub.notebook_output_download('mohamedhassanali/apple-vs-tomatoes-binaryclassify-pytorch')

print('Data source import complete.')


100%|██████████| 545M/545M [00:09<00:00, 63.4MB/s]

Extracting files...


Extracting files...
Data source import complete.


In [2]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
!pip install tensorflow

In [6]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator


IMAGE_DIR = ashfakyeafi_cat_dog_images_for_classification_path + '/cat_dog'


filenames = os.listdir(IMAGE_DIR)
categories = []
for filename in filenames:
    if filename.startswith('dog'):
        categories.append('dog')
    else:
        categories.append('cat')

df = pd.DataFrame({
    'filename': filenames,
    'category': categories
})


df = df.sample(n=25000, random_state=42).reset_index(drop=True)


train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True
)


val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

IMAGE_SIZE = (150, 150)
BATCH_SIZE = 32

train_generator = train_datagen.flow_from_dataframe(
    dataframe=df,
    directory=IMAGE_DIR,
    x_col='filename',
    y_col='category',
    target_size=IMAGE_SIZE,
    class_mode='binary',
    batch_size=BATCH_SIZE,
    subset='training',
    seed=123
)

val_generator = val_datagen.flow_from_dataframe(
    dataframe=df,
    directory=IMAGE_DIR,
    x_col='filename',
    y_col='category',
    target_size=IMAGE_SIZE,
    class_mode='binary',
    batch_size=BATCH_SIZE,
    subset='validation',
    seed=123
)


model = models.Sequential([

    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),


    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),


    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),


    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])


model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


Found 20000 validated image filenames belonging to 2 classes.
Found 5000 validated image filenames belonging to 2 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
print("Starting training...")
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30
)
print("Training complete!")

Starting training...
Epoch 1/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 156s 236ms/step - accuracy: 0.5985 - loss: 0.6613 - val_accuracy: 0.7206 - val_loss: 0.5753
Epoch 2/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 139s 223ms/step - accuracy: 0.6762 - loss: 0.5991 - val_accuracy: 0.6876 - val_loss: 0.5828
Epoch 3/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 137s 220ms/step - accuracy: 0.7063 - loss: 0.5624 - val_accuracy: 0.7578 - val_loss: 0.4970
Epoch 4/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 140s 224ms/step - accuracy: 0.7337 - loss: 0.5303 - val_accuracy: 0.7806 - val_loss: 0.4633
Epoch 5/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 139s 223ms/step - accuracy: 0.7581 - loss: 0.5014 - val_accuracy: 0.7932 - val_loss: 0.4523
Epoch 6/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 142s 228ms/step - accuracy: 0.7727 - loss: 0.4797 - val_accuracy: 0.8016 - val_loss: 0.4425
Epoch 7/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 143s 229ms/step - accuracy: 0.7804 - loss: 0.4640 - val_accuracy: 0.8220 - val_loss: 0.3990
Epoch 8/30
625/625 ━━━━━━━━━━━━━━━━━━━━ 143s 228ms/step

In [8]:
final_train_acc = history.history['accuracy'][-1] * 100
final_val_acc = history.history['val_accuracy'][-1] * 100

print("-" * 40)
print(f"Final Training Accuracy:   {final_train_acc:.2f}%")
print(f"Final Validation Accuracy: {final_val_acc:.2f}%")
print("-" * 40)

----------------------------------------
Final Training Accuracy:   87.31%
Final Validation Accuracy: 90.92%
----------------------------------------
